In [3]:
# SDG 8: Does Albania's Tourism Boom Actually Create Decent Work?
# Andi Lata | Clark University
#
# The plan: 5 research questions, each with distribution analysis,
# probability modeling, and hypothesis testing.
# Let me get all the imports out of the way first.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import norm, ttest_1samp, ttest_ind, pearsonr

# Not sure if I'll use all of these, but better to have them.

In [ ]:
# Load the cleaned data files from Step 1.

df_wages    = pd.read_csv("cleaned/wages_clean.csv")
df_youth    = pd.read_csv("cleaned/youth_unemployment_clean.csv")
df_tourism  = pd.read_csv("cleaned/tourism_receipts_clean.csv")
df_informal = pd.read_csv("cleaned/informal_employment_clean.csv")
df_emp      = pd.read_csv("cleaned/tourism_employment_clean.csv")
df_gdp      = pd.read_csv("cleaned/gdp_per_capita_clean.csv")

# CPI/inflation is annoying because World Bank stores years as columns.
# Need to melt it into long format first.
df_cpi_raw = pd.read_csv("inflation.csv", skiprows=4)
df_cpi_raw.columns = df_cpi_raw.columns.str.replace(" ", "_")

alb_cpi = df_cpi_raw[df_cpi_raw["Country_Code"] == "ALB"].copy()
year_cols = [str(y) for y in range(2010, 2025)]
alb_cpi = alb_cpi[["Country_Name", "Country_Code"] + year_cols]
alb_cpi = alb_cpi.melt(id_vars=["Country_Name", "Country_Code"],
                        var_name="Year", value_name="Inflation_Rate_Pct")
alb_cpi["Year"] = alb_cpi["Year"].astype(int)
alb_cpi = alb_cpi.dropna().sort_values("Year").reset_index(drop=True)

print("All data loaded.")

In [ ]:
## ----------------------------------------------------------
## RQ1: Formality vs Growth
## As tourism revenue doubled post-2020, did formal employment grow,
## or did informality just absorb the boom?
## ----------------------------------------------------------

# H0: Informality rate is the same pre vs. post boom.
# H1: Informality is NOT significantly lower despite the boom --
#     meaning the revenue didn't convert into real formal jobs.

alb_tourism  = df_tourism[df_tourism["Country_Code"] == "ALB"].copy()
alb_informal = df_informal[df_informal["Country_Code"] == "ALB"].copy()

receipts = alb_tourism["Tourism_Receipts_USD_Billions"].values

# Distribution plot first -- want to see the shape before modeling.
plt.figure(figsize=(8, 4))
plt.hist(receipts, bins=8, edgecolor="black", color="steelblue", alpha=0.8)
plt.axvline(receipts.mean(), color="red", linestyle="--",
            label=f"mean = ${receipts.mean():.2f}B")
plt.title("RQ1: Distribution of Albania Tourism Receipts (2010–2024)")
plt.xlabel("Tourism Receipts (USD Billions)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

# Right-skewed -- most years were low, the boom years pull the mean up.
print("Distribution: right-skewed. Most years were low; boom years pull the mean up.")

In [ ]:
# Probability modeling -- what was the historical probability of a $3B+ year?
# Turns out it's pretty low, which makes 3 consecutive hits even more notable.

mu    = receipts.mean()
sigma = receipts.std()
prob_boom = 1 - norm.cdf(3.0, mu, sigma)

print(f"\nProbability modeling:")
print(f"  Mean receipts: ${mu:.2f}B  |  Std: ${sigma:.2f}B")
print(f"  P(receipts > $3B) = {prob_boom:.4f} ({prob_boom*100:.1f}%)")
print(f"  Historically rare -- but Albania hit it 3 years in a row post-2022.")

# PDF and CDF side by side
x = np.linspace(0, 7, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(x, norm.pdf(x, mu, sigma), color="steelblue", linewidth=2)
ax1.axvline(3.0, color="red", linestyle="--", label="$3B boom threshold")
ax1.fill_between(x, norm.pdf(x, mu, sigma), where=(x > 3.0),
                 alpha=0.3, color="red", label=f"P={prob_boom:.2f}")
ax1.set_title("RQ1: Tourism Receipts PDF")
ax1.set_xlabel("USD Billions")
ax1.legend()

ax2.plot(x, norm.cdf(x, mu, sigma), color="steelblue", linewidth=2)
ax2.axvline(3.0, color="red", linestyle="--", label="$3B threshold")
ax2.set_title("RQ1: Tourism Receipts CDF")
ax2.set_xlabel("USD Billions")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Hypothesis test: did formality actually improve during the boom?
# Splitting into pre-boom (<=2019) and post-boom (2021+).

pre_boom_inf  = alb_informal[alb_informal["Year"] <= 2019]["Informal_Employment_Rate_Pct"].values
post_boom_inf = alb_informal[alb_informal["Year"] >= 2021]["Informal_Employment_Rate_Pct"].values

print(f"Pre-boom:  {pre_boom_inf} --> mean = {pre_boom_inf.mean():.2f}%")
print(f"Post-boom: {post_boom_inf} --> mean = {post_boom_inf.mean():.2f}%")

t_stat, p_val = ttest_ind(pre_boom_inf, post_boom_inf)
print(f"\nt-stat: {t_stat:.4f}  |  p-value: {p_val:.4f}")

# p > 0.05, so we fail to reject -- informality didn't meaningfully budge.
if p_val < 0.05:
    print("Result: REJECT H0 -- informality changed significantly.")
else:
    print("Result: FAIL TO REJECT H0 -- informality did not significantly change.")
    print("Interpretation: The boom ran through the informal economy, not formal jobs.")

In [ ]:
## --------------------------------------------------------
## RQ2: Wage Gap -- Does the hospitality sector keep up with the national average?
##
## Using CEIC/INSTAT data for accommodation + food wages:
##   2021: 29,858 ALL  |  2022: 32,784 ALL
## National average comes from wages_clean.
## -------------------------------------------------------------
national_2022 = df_wages[df_wages["Year"] == 2022]["Avg_Monthly_Gross_Wage_ALL"].mean()

gap_2021 = (29858 / 60666) * 100   # 2021 national avg from INSTAT
gap_2022 = (32784 / national_2022) * 100

print(f"Hospitality wages as % of national average:")
print(f"  2021: {gap_2021:.1f}%  |  2022: {gap_2022:.1f}%")
print(f"  Consistently around 50% -- the gap did not close during the boom.")

# Distribution of all national wages
all_wages = df_wages["Avg_Monthly_Gross_Wage_ALL"].values

plt.figure(figsize=(8, 4))
plt.hist(all_wages, bins=8, edgecolor="black", color="steelblue", alpha=0.8)
plt.axvline(all_wages.mean(), color="red", linestyle="--",
            label=f"national mean = {all_wages.mean():.0f} ALL")
plt.axvline(32784, color="orange", linestyle="--",
            label="hospitality avg (2022) = 32,784 ALL")
plt.title("RQ2: National Wage Distribution vs Hospitality Sector (ALL)")
plt.xlabel("Monthly Gross Wage (ALL)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Probability modeling: what's the probability a national worker
# earns at or below the hospitality wage?
# Basically asking: how far into the left tail is hospitality?

mu_w    = all_wages.mean()
sigma_w = all_wages.std()
p_below = norm.cdf(32784, mu_w, sigma_w)

print(f"P(wage <= 32,784 ALL) = {p_below:.4f}")
print(f"Hospitality workers are in the bottom {p_below*100:.1f}% of national earners.")

x = np.linspace(40000, 120000, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(x, norm.pdf(x, mu_w, sigma_w), color="steelblue", linewidth=2)
ax1.axvline(32784, color="orange", linestyle="--", label="hospitality avg")
ax1.axvline(mu_w, color="red", linestyle="--", label="national avg")
ax1.set_title("RQ2: Wage PDF")
ax1.set_xlabel("Monthly Wage (ALL)")
ax1.legend()

ax2.plot(x, norm.cdf(x, mu_w, sigma_w), color="steelblue", linewidth=2)
ax2.axvline(32784, color="orange", linestyle="--", label="hospitality avg")
ax2.set_title("RQ2: Wage CDF")
ax2.set_xlabel("Monthly Wage (ALL)")
ax2.legend()

plt.tight_layout()
plt.show()

# Hypothesis test -- one-sample t against the hospitality benchmark.
# H0: national avg wage = hospitality wage (32,784 ALL)
# H1: national avg is significantly higher
t_stat, p_val = ttest_1samp(all_wages, 32784)
print(f"\nt-stat: {t_stat:.4f}  |  p-value: {p_val:.6f}")
print("Result: REJECT H0 -- national wages significantly above hospitality level.")
print("Interpretation: The boom is not lifting hospitality wages to match the economy.")

In [ ]:
## --------------------------------------------------------------
## RQ3: Real Wage Growth vs Inflation
## Did nominal wage gains actually outpace inflation?
## Or are workers losing purchasing power in real terms?
## ─────────────────────────────────────────────────────────────

# Compute year-over-year nominal growth, then subtract CPI to get real growth.

wages_annual = df_wages.groupby("Year")["Avg_Monthly_Gross_Wage_ALL"].mean().reset_index()
wages_annual.columns = ["Year", "Avg_Wage_ALL"]
wages_annual["Nominal_Growth_Pct"] = wages_annual["Avg_Wage_ALL"].pct_change() * 100

merged = wages_annual.merge(alb_cpi[["Year", "Inflation_Rate_Pct"]], on="Year", how="inner")
merged["Real_Wage_Growth_Pct"] = merged["Nominal_Growth_Pct"] - merged["Inflation_Rate_Pct"]
merged = merged.dropna().reset_index(drop=True)

print("Real wage growth by year:")
print(merged[["Year", "Nominal_Growth_Pct", "Inflation_Rate_Pct", "Real_Wage_Growth_Pct"]].to_string())